# 🎵 Phase 1A-Complete: Full Data Pipeline with Augmentation, Scaling & Imbalance Handling

**ต่อจาก Phase 1A (data pipeline) + Phase 1A-EDA (cleaning)**

Notebook นี้เพิ่ม 3 ส่วนที่ขาด:

| # | Component | วิธีการ | ทำตอนไหน |
|---|-----------|---------|----------|
| 1 | **Data Augmentation** | SpecAugment + Noise + Time Stretch + Pitch Shift | On-the-fly (training only) |
| 2 | **Feature Scaling** | StandardScaler (tabular) + Global Normalization (mel) | Fit on train → apply all |
| 3 | **Class Imbalance** | WeightedRandomSampler + class weights | DataLoader + Loss |

**Output:** Updated `FMADatasetV2` class ที่รวมทุกอย่าง + saved scaler/stats files


## 0. Setup

In [ ]:
import os
import json
import pickle
import warnings
import logging
from pathlib import Path
from typing import Tuple, Optional, Dict, List

import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

# ============================================================
# PATHS & CONFIG
# ============================================================
FMA_AUDIO_DIR = Path(r"D:\patt\project\pattern-music\FMA_Data\fma_small\fma_small")
FMA_METADATA_CSV = Path(r"D:\patt\project\pattern-music\FMA_Data\fma_metadata\fma_metadata\tracks.csv")

SAMPLE_RATE = 22050
CHUNK_DURATION = 3.0
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
TARGET_TIME_FRAMES = 130
MEL_SHAPE = (1, N_MELS, TARGET_TIME_FRAMES)

BATCH_SIZE = 32
NUM_WORKERS = 4
PIN_MEMORY = True

CORRUPTED_TRACK_IDS = {98565, 98567, 98569, 99134, 108925, 133297, 143992}

GENRE_LABELS = ['Electronic', 'Experimental', 'Folk', 'Hip-Hop',
                'Instrumental', 'International', 'Pop', 'Rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRE_LABELS)}
NUM_CLASSES = len(GENRE_LABELS)

OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Setup complete")


## 1. Load Metadata (from EDA or fresh)

In [ ]:
def load_fma_tracks(csv_path: Path) -> pd.DataFrame:
    """โหลด tracks.csv multi-level headers, filter fma_small, clean"""
    tracks = pd.read_csv(csv_path, index_col=0, header=[0, 1])
    small_mask = tracks[('set', 'subset')] == 'small'
    df = tracks.loc[small_mask, [('track', 'genre_top'), ('set', 'split')]].copy()
    df.columns = ['genre_top', 'split']
    df.index.name = 'track_id'
    df = df.dropna(subset=['genre_top'])
    df = df.drop(index=list(CORRUPTED_TRACK_IDS.intersection(set(df.index))), errors='ignore')
    df['genre_idx'] = df['genre_top'].map(GENRE_TO_IDX)
    return df.reset_index()

def get_audio_path(audio_dir: Path, track_id: int) -> Path:
    tid_str = f"{track_id:06d}"
    return audio_dir / tid_str[:3] / f"{tid_str}.mp3"

# ถ้ามี cleaned_metadata.csv จาก EDA ให้ใช้อันนั้น (สะอาดกว่า)
cleaned_path = OUTPUT_DIR / 'cleaned_metadata.csv'
if cleaned_path.exists():
    metadata_df = pd.read_csv(cleaned_path)
    print(f"✓ Loaded cleaned metadata: {len(metadata_df)} tracks")
else:
    metadata_df = load_fma_tracks(FMA_METADATA_CSV)
    print(f"✓ Loaded raw metadata: {len(metadata_df)} tracks")
    print("  ⚠️ Run EDA notebook first for cleaner data!")

print(f"  Splits: {metadata_df['split'].value_counts().to_dict()}")
print(f"  Genres: {metadata_df['genre_top'].nunique()}")


## 2. 🔊 Data Augmentation

### ทำไมต้อง Augment?
FMA small มีแค่ ~6,400 training tracks — น้อยสำหรับ deep learning
Augmentation ช่วยสร้าง "virtual examples" ที่หลากหลายขึ้น ลด overfitting

### Strategy:
| Augmentation | ทำกับอะไร | Effect | Probability |
|--------------|----------|--------|-------------|
| **Random Chunk** | raw audio | Temporal augmentation — เห็นส่วนต่างๆ ของเพลง | 100% (always) |
| **Gaussian Noise** | raw audio | Robustness ต่อ noise | 30% |
| **Time Stretch** | raw audio | เร่ง/ช้าลง ±10% — ไม่เปลี่ยน pitch | 20% |
| **Pitch Shift** | raw audio | เลื่อน pitch ±2 semitones — ไม่เปลี่ยน tempo | 20% |
| **SpecAugment** | mel spectrogram | Mask frequency/time bands — สำคัญมาก | 50% |

⚠️ ทำเฉพาะ **training** เท่านั้น — val/test ต้อง deterministic


In [ ]:
class AudioAugmentor:
    """
    Audio-level augmentation สำหรับ training
    ทุก method รับ 1D numpy array แล้ว return 1D numpy array (same length)
    """

    def __init__(self, sr: int = SAMPLE_RATE, seed: Optional[int] = None):
        self.sr = sr
        self.rng = np.random.RandomState(seed)

    def add_gaussian_noise(self, audio: np.ndarray, snr_db: float = None) -> np.ndarray:
        """
        เพิ่ม Gaussian noise ด้วย Signal-to-Noise Ratio ที่กำหนด
        SNR สูง = noise น้อย, SNR ต่ำ = noise เยอะ

        Args:
            audio: 1D audio array
            snr_db: Signal-to-Noise Ratio in dB (default: random 15-30 dB)
        """
        if snr_db is None:
            snr_db = self.rng.uniform(15, 30)  # เบาๆ พอ — ไม่ทำลาย signal

        # คำนวณ noise level จาก SNR
        signal_power = np.mean(audio ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = self.rng.normal(0, np.sqrt(noise_power), len(audio)).astype(np.float32)

        return audio + noise

    def time_stretch(self, audio: np.ndarray, rate: float = None) -> np.ndarray:
        """
        เร่ง/ช้า audio โดยไม่เปลี่ยน pitch
        rate > 1 = เร็วขึ้น, rate < 1 = ช้าลง

        จะ crop/pad ให้ output length เท่าเดิม
        """
        if rate is None:
            rate = self.rng.uniform(0.9, 1.1)  # ±10%

        stretched = librosa.effects.time_stretch(audio, rate=rate)

        # Crop or pad to original length
        original_len = len(audio)
        if len(stretched) > original_len:
            stretched = stretched[:original_len]
        elif len(stretched) < original_len:
            padded = np.zeros(original_len, dtype=np.float32)
            padded[:len(stretched)] = stretched
            stretched = padded

        return stretched.astype(np.float32)

    def pitch_shift(self, audio: np.ndarray, n_steps: float = None) -> np.ndarray:
        """
        เลื่อน pitch ขึ้น/ลง n semitones โดยไม่เปลี่ยน tempo
        """
        if n_steps is None:
            n_steps = self.rng.uniform(-2, 2)  # ±2 semitones

        shifted = librosa.effects.pitch_shift(y=audio, sr=self.sr, n_steps=n_steps)
        return shifted.astype(np.float32)

    def apply(self, audio: np.ndarray,
              p_noise: float = 0.3,
              p_stretch: float = 0.2,
              p_pitch: float = 0.2) -> np.ndarray:
        """
        Apply augmentation chain ด้วย probability ที่กำหนด
        แต่ละ augmentation เป็น independent — อาจเกิดหลายอันพร้อมกัน
        """
        aug_audio = audio.copy()

        if self.rng.random() < p_noise:
            aug_audio = self.add_gaussian_noise(aug_audio)

        if self.rng.random() < p_stretch:
            aug_audio = self.time_stretch(aug_audio)

        if self.rng.random() < p_pitch:
            aug_audio = self.pitch_shift(aug_audio)

        return aug_audio


class SpecAugment:
    """
    SpecAugment (Google, 2019) — mask random bands บน mel spectrogram

    2 ประเภท:
    1. Frequency Masking — mask random frequency bands (แนวนอน)
       ช่วยให้ model ไม่ rely on specific frequency range เกินไป
    2. Time Masking — mask random time steps (แนวตั้ง)
       ช่วยให้ model robust ต่อ missing segments

    Apply หลัง mel spectrogram extraction, ก่อน normalization
    """

    def __init__(self, freq_mask_param: int = 15, time_mask_param: int = 20,
                 n_freq_masks: int = 2, n_time_masks: int = 2,
                 seed: Optional[int] = None):
        """
        Args:
            freq_mask_param: max width of frequency mask (จาก 128 mel bins)
            time_mask_param: max width of time mask (จาก 130 frames)
            n_freq_masks: จำนวน frequency masks ที่จะ apply
            n_time_masks: จำนวน time masks ที่จะ apply
        """
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        self.n_freq_masks = n_freq_masks
        self.n_time_masks = n_time_masks
        self.rng = np.random.RandomState(seed)

    def apply(self, mel_spec: np.ndarray) -> np.ndarray:
        """
        Apply SpecAugment บน mel spectrogram

        Args:
            mel_spec: shape (1, n_mels, n_frames) — log-mel spectrogram
        Returns:
            augmented mel spectrogram (same shape)
        """
        aug = mel_spec.copy()
        _, n_mels, n_frames = aug.shape

        # Frequency masking
        for _ in range(self.n_freq_masks):
            f = self.rng.randint(0, self.freq_mask_param + 1)
            f0 = self.rng.randint(0, max(1, n_mels - f))
            aug[:, f0:f0 + f, :] = 0  # mask ด้วย 0

        # Time masking
        for _ in range(self.n_time_masks):
            t = self.rng.randint(0, self.time_mask_param + 1)
            t0 = self.rng.randint(0, max(1, n_frames - t))
            aug[:, :, t0:t0 + t] = 0  # mask ด้วย 0

        return aug

print("✓ Augmentation classes defined")


### 2.1 Visualize Augmentation Effects

In [ ]:
# โหลด 1 เพลง แล้วเปรียบเทียบ original vs augmented
from fma_phase1a_data_pipeline import extract_mel_spectrogram, extract_tabular_features

test_row = metadata_df.iloc[0]
test_path = get_audio_path(FMA_AUDIO_DIR, test_row['track_id'])
audio, sr = librosa.load(test_path, sr=SAMPLE_RATE, mono=True)

# Center chunk
start = (len(audio) - CHUNK_SAMPLES) // 2
chunk = audio[start:start + CHUNK_SAMPLES]

# Create augmentors
audio_aug = AudioAugmentor(seed=42)
spec_aug = SpecAugment(seed=42)

# Generate augmented versions
chunk_noise = audio_aug.add_gaussian_noise(chunk, snr_db=20)
chunk_stretch = audio_aug.time_stretch(chunk, rate=0.9)
chunk_pitch = audio_aug.pitch_shift(chunk, n_steps=2)
chunk_combined = audio_aug.apply(chunk, p_noise=1.0, p_stretch=1.0, p_pitch=1.0)

# Mel spectrograms
mel_original = extract_mel_spectrogram(chunk, SAMPLE_RATE)
mel_noise = extract_mel_spectrogram(chunk_noise, SAMPLE_RATE)
mel_stretch = extract_mel_spectrogram(chunk_stretch, SAMPLE_RATE)
mel_pitch = extract_mel_spectrogram(chunk_pitch, SAMPLE_RATE)
mel_specaug = spec_aug.apply(mel_original)

# Plot all
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
specs = [
    (mel_original, 'Original'),
    (mel_noise, 'Gaussian Noise (SNR=20dB)'),
    (mel_stretch, 'Time Stretch (0.9x)'),
    (mel_pitch, 'Pitch Shift (+2 semitones)'),
    (mel_specaug, 'SpecAugment'),
    (extract_mel_spectrogram(chunk_combined, SAMPLE_RATE), 'All Combined'),
]

for i, (mel, title) in enumerate(specs):
    ax = axes[i // 3, i % 3]
    ax.imshow(mel[0], aspect='auto', origin='lower', cmap='magma',
              extent=[0, CHUNK_DURATION, 0, SAMPLE_RATE // 2])
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.set_ylabel('Freq (Hz)', fontsize=9)

plt.suptitle(f'Augmentation Effects — Genre: {test_row["genre_top"]} (Track {test_row["track_id"]})',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'augmentation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Original vs Augmented spectrograms plotted")
print("  SpecAugment = black bands (masked regions)")
print("  Noise/Stretch/Pitch = subtle texture changes")


## 3. 📐 Feature Scaling

### Strategy:
| Data Type | Method | ทำไม |
|-----------|--------|------|
| **Tabular (88 features)** | `StandardScaler` (zero mean, unit variance) | Features มี scale ต่างกันมาก (MFCC ~-100 to 100, RMS ~0.001 to 0.5) |
| **Mel Spectrogram** | Global Normalization `(x - mean) / std` | dB values range ~-80 to 0 — normalize ให้ ~N(0,1) |

⚠️ **Critical:** fit scaler เฉพาะ training set → transform val/test ด้วย scaler เดียวกัน
ป้องกัน data leakage


In [ ]:
def compute_scaling_stats(metadata_df: pd.DataFrame, audio_dir: Path,
                          max_samples: int = 2000) -> dict:
    """
    คำนวณ scaling statistics จาก training set เท่านั้น

    Returns dict ที่มี:
      - tabular_scaler: fitted StandardScaler
      - mel_mean: global mean ของ log-mel spectrogram
      - mel_std: global std ของ log-mel spectrogram

    ใช้ subset (max_samples) เพื่อประหยัดเวลา — stats จะ converge ที่ ~1000-2000 samples
    """
    train_df = metadata_df[metadata_df['split'] == 'training']

    # Sample สำหรับ compute stats (ไม่จำเป็นต้องใช้ทุก track)
    if len(train_df) > max_samples:
        train_sample = train_df.sample(n=max_samples, random_state=42)
    else:
        train_sample = train_df

    logger.info(f"Computing scaling stats from {len(train_sample)} training tracks...")

    all_tabular = []
    all_mel_means = []
    all_mel_stds = []
    errors = 0

    for _, row in tqdm(train_sample.iterrows(), total=len(train_sample), desc="Computing stats"):
        track_id = row['track_id']
        audio_path = get_audio_path(audio_dir, track_id)

        try:
            audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

            # Center chunk
            if len(audio) >= CHUNK_SAMPLES:
                start = (len(audio) - CHUNK_SAMPLES) // 2
                chunk = audio[start:start + CHUNK_SAMPLES]
            else:
                chunk = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
                chunk[:len(audio)] = audio

            # Tabular features
            tab = extract_tabular_features(chunk, SAMPLE_RATE)
            all_tabular.append(tab)

            # Mel spectrogram
            mel = extract_mel_spectrogram(chunk, SAMPLE_RATE)
            all_mel_means.append(mel.mean())
            all_mel_stds.append(mel.std())

        except Exception as e:
            errors += 1

    # --- Fit StandardScaler บน tabular features ---
    tabular_matrix = np.array(all_tabular)  # (N, 88)
    scaler = StandardScaler()
    scaler.fit(tabular_matrix)

    # --- Global mel normalization stats ---
    mel_mean = float(np.mean(all_mel_means))
    mel_std = float(np.mean(all_mel_stds))

    stats = {
        'tabular_scaler': scaler,
        'mel_mean': mel_mean,
        'mel_std': mel_std,
        'n_samples_used': len(all_tabular),
        'tabular_feature_means': scaler.mean_.tolist(),
        'tabular_feature_stds': scaler.scale_.tolist(),
    }

    logger.info(f"✓ Stats computed from {len(all_tabular)} tracks ({errors} errors)")
    logger.info(f"  Mel global mean: {mel_mean:.4f}, std: {mel_std:.4f}")
    logger.info(f"  Tabular means range: [{scaler.mean_.min():.2f}, {scaler.mean_.max():.2f}]")
    logger.info(f"  Tabular stds range:  [{scaler.scale_.min():.4f}, {scaler.scale_.max():.2f}]")

    return stats

print("✓ compute_scaling_stats() defined")


In [ ]:
# คำนวณ scaling stats (ใช้เวลา ~5-10 นาที)
scaling_stats = compute_scaling_stats(metadata_df, FMA_AUDIO_DIR, max_samples=2000)

# Save scaler & stats
with open(OUTPUT_DIR / 'tabular_scaler.pkl', 'wb') as f:
    pickle.dump(scaling_stats['tabular_scaler'], f)

scaling_json = {
    'mel_mean': scaling_stats['mel_mean'],
    'mel_std': scaling_stats['mel_std'],
    'n_samples_used': scaling_stats['n_samples_used'],
    'tabular_feature_means': scaling_stats['tabular_feature_means'],
    'tabular_feature_stds': scaling_stats['tabular_feature_stds'],
}
with open(OUTPUT_DIR / 'scaling_stats.json', 'w') as f:
    json.dump(scaling_json, f, indent=2)

print(f"\n✓ Saved tabular_scaler.pkl")
print(f"✓ Saved scaling_stats.json")
print(f"  Mel mean: {scaling_stats['mel_mean']:.4f}")
print(f"  Mel std:  {scaling_stats['mel_std']:.4f}")


### 3.1 Verify Scaling

In [ ]:
# ทดสอบ scaling กับ 1 sample
test_tab = extract_tabular_features(chunk, SAMPLE_RATE)
test_mel = extract_mel_spectrogram(chunk, SAMPLE_RATE)

# Scale tabular
scaler = scaling_stats['tabular_scaler']
test_tab_scaled = scaler.transform(test_tab.reshape(1, -1))[0]

# Scale mel
mel_mean = scaling_stats['mel_mean']
mel_std = scaling_stats['mel_std']
test_mel_scaled = (test_mel - mel_mean) / mel_std

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Tabular before/after
axes[0, 0].bar(range(len(test_tab)), test_tab, color='#2196F3', alpha=0.7)
axes[0, 0].set_title('Tabular Features — Before Scaling')
axes[0, 0].set_xlabel('Feature Index')
axes[0, 0].set_ylabel('Value')

axes[0, 1].bar(range(len(test_tab_scaled)), test_tab_scaled, color='#4CAF50', alpha=0.7)
axes[0, 1].set_title('Tabular Features — After StandardScaler')
axes[0, 1].set_xlabel('Feature Index')
axes[0, 1].set_ylabel('Scaled Value')
axes[0, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Mel before/after
axes[1, 0].imshow(test_mel[0], aspect='auto', origin='lower', cmap='magma')
axes[1, 0].set_title(f'Mel Spectrogram — Before (range: {test_mel.min():.1f} to {test_mel.max():.1f})')

axes[1, 1].imshow(test_mel_scaled[0], aspect='auto', origin='lower', cmap='magma')
axes[1, 1].set_title(f'Mel Spectrogram — After (range: {test_mel_scaled.min():.1f} to {test_mel_scaled.max():.1f})')

plt.suptitle('Feature Scaling Verification', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scaling_verification.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Tabular before — mean: {test_tab.mean():.2f}, std: {test_tab.std():.2f}")
print(f"Tabular after  — mean: {test_tab_scaled.mean():.2f}, std: {test_tab_scaled.std():.2f}")
print(f"Mel before     — mean: {test_mel.mean():.2f}, std: {test_mel.std():.2f}")
print(f"Mel after      — mean: {test_mel_scaled.mean():.2f}, std: {test_mel_scaled.std():.2f}")


## 4. ⚖️ Class Imbalance Handling

### WeightedRandomSampler vs Class Weights

| Method | ทำยังไง | ข้อดี | ข้อเสีย |
|--------|---------|-------|---------|
| **WeightedRandomSampler** | สุ่ม sample ให้ทุก genre มี probability เท่ากัน | ทุก epoch เห็น genre เท่าๆ กัน | ไม่ 100% balanced ทุก batch |
| **Class Weights in Loss** | ให้ loss ของ minority class มี weight สูงขึ้น | ง่าย, ทุก sample ถูกใช้ | ไม่แก้ปัญหา sampling bias |

**เราจะทำทั้ง 2 วิธี** — ให้ user เลือกตอน training ว่าจะใช้อันไหน


In [ ]:
def compute_class_weights(metadata_df: pd.DataFrame) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    คำนวณ class weights 2 แบบ:
    1. class_weights: inverse frequency — สำหรับ nn.CrossEntropyLoss(weight=...)
    2. sample_weights: weight ต่อ sample — สำหรับ WeightedRandomSampler

    Returns:
        class_weights: FloatTensor shape (NUM_CLASSES,)
        sample_weights: FloatTensor shape (len(training_set),)
    """
    train_df = metadata_df[metadata_df['split'] == 'training'].reset_index(drop=True)

    # Count per genre
    genre_counts = train_df['genre_top'].value_counts()
    total = len(train_df)

    # --- Class Weights (inverse frequency) ---
    weights = []
    for genre in GENRE_LABELS:
        count = genre_counts.get(genre, 1)
        # inverse frequency: total / (num_classes * count)
        w = total / (NUM_CLASSES * count)
        weights.append(w)

    class_weights = torch.FloatTensor(weights)

    # --- Sample Weights (for WeightedRandomSampler) ---
    # แต่ละ sample ได้ weight = class_weight ของ genre นั้น
    genre_to_weight = {genre: class_weights[i].item() for i, genre in enumerate(GENRE_LABELS)}
    sample_weights = torch.FloatTensor([
        genre_to_weight[row['genre_top']] for _, row in train_df.iterrows()
    ])

    return class_weights, sample_weights

class_weights, sample_weights = compute_class_weights(metadata_df)

print("Class Weights (for CrossEntropyLoss):")
print("-" * 45)
for i, genre in enumerate(GENRE_LABELS):
    train_count = len(metadata_df[(metadata_df['genre_top'] == genre) & (metadata_df['split'] == 'training')])
    print(f"  [{i}] {genre:15s}: weight={class_weights[i]:.3f} (n={train_count})")

print(f"\nSample weights shape: {sample_weights.shape}")
print(f"Sample weights range: [{sample_weights.min():.3f}, {sample_weights.max():.3f}]")


In [ ]:
# Visualize: original distribution vs what WeightedRandomSampler จะ sample
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_df = metadata_df[metadata_df['split'] == 'training']

# Original
genre_counts = train_df['genre_top'].value_counts().sort_index()
genre_counts.plot(kind='bar', ax=axes[0], color='#F44336', edgecolor='white')
axes[0].set_title('Original Training Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(genre_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=9)

# Simulated WeightedRandomSampler (1 epoch worth of samples)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
sampled_indices = list(sampler)
sampled_genres = [train_df.iloc[i]['genre_top'] for i in sampled_indices]
sampled_counts = pd.Series(sampled_genres).value_counts().sort_index()
sampled_counts.plot(kind='bar', ax=axes[1], color='#4CAF50', edgecolor='white')
axes[1].set_title('After WeightedRandomSampler (1 epoch)')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(sampled_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=9)

plt.suptitle('Class Imbalance Handling — Before vs After', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_imbalance_handling.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ WeightedRandomSampler ทำให้ทุก genre ถูก sample ~เท่าๆ กัน")


## 5. 🏗️ FMADatasetV2 — Complete Dataset with Everything

Dataset ใหม่ที่รวม:
- ✅ Metadata parsing & cleaning
- ✅ Chunking strategy (random/center)
- ✅ Feature extraction (tabular + mel)
- ✅ Data augmentation (audio-level + SpecAugment)
- ✅ Feature scaling (StandardScaler + global mel normalization)


In [ ]:
class FMADatasetV2(Dataset):
    """
    Complete FMA Dataset with augmentation & scaling

    เปลี่ยนจาก V1:
    1. เพิ่ม audio augmentation (training only)
    2. เพิ่ม SpecAugment (training only)
    3. เพิ่ม feature scaling (tabular + mel)
    4. รับ scaling_stats จากภายนอก (ป้องกัน data leakage)
    """

    def __init__(
        self,
        metadata_df: pd.DataFrame,
        audio_dir: Path,
        split: str = 'training',
        scaling_stats: Optional[dict] = None,
        enable_augmentation: bool = True,
        augment_config: Optional[dict] = None,
        sample_rate: int = SAMPLE_RATE,
        chunk_samples: int = CHUNK_SAMPLES,
    ):
        self.audio_dir = audio_dir
        self.sr = sample_rate
        self.chunk_samples = chunk_samples
        self.split = split
        self.is_training = (split == 'training')
        self.scaling_stats = scaling_stats
        self.enable_augmentation = enable_augmentation and self.is_training

        # Default augmentation config
        self.aug_config = augment_config or {
            'p_noise': 0.3,
            'p_stretch': 0.2,
            'p_pitch': 0.2,
            'p_specaugment': 0.5,
        }

        # Initialize augmentors (training only)
        if self.enable_augmentation:
            self.audio_augmentor = AudioAugmentor(sr=sample_rate)
            self.spec_augmentor = SpecAugment()

        # Scaler สำหรับ tabular features
        self.tabular_scaler = None
        self.mel_mean = None
        self.mel_std = None
        if scaling_stats:
            self.tabular_scaler = scaling_stats.get('tabular_scaler')
            self.mel_mean = scaling_stats.get('mel_mean')
            self.mel_std = scaling_stats.get('mel_std')

        # Filter split & validate
        self.df = metadata_df[metadata_df['split'] == split].reset_index(drop=True)

        valid_indices = []
        for idx, row in self.df.iterrows():
            audio_path = get_audio_path(self.audio_dir, row['track_id'])
            if audio_path.exists() and audio_path.stat().st_size > 0:
                valid_indices.append(idx)

        self.df = self.df.loc[valid_indices].reset_index(drop=True)
        logger.info(f"[{split}] DatasetV2 ready: {len(self.df)} tracks | "
                    f"aug={'ON' if self.enable_augmentation else 'OFF'} | "
                    f"scaling={'ON' if scaling_stats else 'OFF'}")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        track_id = row['track_id']
        genre_idx = row['genre_idx']

        # --- 1. Load MP3 ---
        audio_path = get_audio_path(self.audio_dir, track_id)
        try:
            audio, _ = librosa.load(audio_path, sr=self.sr, mono=True)
        except Exception as e:
            logger.error(f"Failed to load track {track_id}: {e}")
            audio = np.zeros(self.chunk_samples, dtype=np.float32)

        # --- 2. Extract Chunk (random/center) ---
        chunk = self._extract_chunk(audio)

        # --- 3. Audio Augmentation (training only) ---
        if self.enable_augmentation:
            chunk = self.audio_augmentor.apply(
                chunk,
                p_noise=self.aug_config['p_noise'],
                p_stretch=self.aug_config['p_stretch'],
                p_pitch=self.aug_config['p_pitch'],
            )

        # --- 4. Feature Extraction ---
        tabular = extract_tabular_features(chunk, self.sr)
        spectrogram = extract_mel_spectrogram(chunk, self.sr)

        # --- 5. SpecAugment (training only, applied on mel spectrogram) ---
        if self.enable_augmentation and np.random.random() < self.aug_config['p_specaugment']:
            spectrogram = self.spec_augmentor.apply(spectrogram)

        # --- 6. Feature Scaling ---
        if self.tabular_scaler is not None:
            tabular = self.tabular_scaler.transform(tabular.reshape(1, -1))[0].astype(np.float32)

        if self.mel_mean is not None and self.mel_std is not None:
            spectrogram = ((spectrogram - self.mel_mean) / self.mel_std).astype(np.float32)

        # --- 7. Convert to tensors ---
        return (
            torch.from_numpy(tabular),
            torch.from_numpy(spectrogram),
            torch.tensor(genre_idx, dtype=torch.long),
        )

    def _extract_chunk(self, audio: np.ndarray) -> np.ndarray:
        total_samples = len(audio)
        if total_samples < self.chunk_samples:
            padded = np.zeros(self.chunk_samples, dtype=np.float32)
            padded[:total_samples] = audio
            return padded
        if self.is_training:
            max_start = total_samples - self.chunk_samples
            start = np.random.randint(0, max_start + 1)
        else:
            start = (total_samples - self.chunk_samples) // 2
        return audio[start:start + self.chunk_samples]

print("✓ FMADatasetV2 defined")


## 6. 🔧 Complete DataLoader Factory

In [ ]:
def create_dataloaders_v2(
    metadata_df: pd.DataFrame,
    audio_dir: Path,
    scaling_stats: dict,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
    use_weighted_sampler: bool = True,
    enable_augmentation: bool = True,
) -> Dict[str, DataLoader]:
    """
    สร้าง DataLoaders ที่ครบทุก component:
    - Augmentation (training only)
    - Scaling (all splits)
    - WeightedRandomSampler (training only, optional)

    Returns:
        dict ของ {split_name: DataLoader}
    """
    loaders = {}

    for split in ['training', 'validation', 'test']:
        dataset = FMADatasetV2(
            metadata_df, audio_dir, split=split,
            scaling_stats=scaling_stats,
            enable_augmentation=(enable_augmentation and split == 'training'),
        )

        # WeightedRandomSampler สำหรับ training
        sampler = None
        shuffle = (split == 'training')

        if split == 'training' and use_weighted_sampler:
            _, sample_w = compute_class_weights(metadata_df)
            # Re-compute สำหรับ filtered dataset (valid files only)
            genre_to_weight = {}
            train_full = metadata_df[metadata_df['split'] == 'training']
            for genre in GENRE_LABELS:
                count = len(train_full[train_full['genre_top'] == genre])
                genre_to_weight[genre] = len(train_full) / (NUM_CLASSES * max(count, 1))

            sw = torch.FloatTensor([
                genre_to_weight[dataset.df.iloc[i]['genre_top']]
                for i in range(len(dataset))
            ])
            sampler = WeightedRandomSampler(sw, num_samples=len(dataset), replacement=True)
            shuffle = False  # sampler และ shuffle ใช้พร้อมกันไม่ได้

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            sampler=sampler,
            num_workers=num_workers,
            pin_memory=PIN_MEMORY,
            drop_last=(split == 'training'),
            persistent_workers=(num_workers > 0),
        )
        loaders[split] = loader

        sampler_status = "WeightedRandomSampler" if sampler else "None"
        logger.info(f"[{split}] DataLoader: {len(dataset)} samples, "
                    f"{len(loader)} batches, sampler={sampler_status}")

    return loaders

print("✓ create_dataloaders_v2() defined")


## 7. ✅ Final Verification

In [ ]:
# สร้าง DataLoaders V2 (num_workers=0 สำหรับ notebook debug)
print("Creating V2 DataLoaders with all components...")

train_dataset = FMADatasetV2(
    metadata_df, FMA_AUDIO_DIR, split='training',
    scaling_stats=scaling_stats, enable_augmentation=True,
)

val_dataset = FMADatasetV2(
    metadata_df, FMA_AUDIO_DIR, split='validation',
    scaling_stats=scaling_stats, enable_augmentation=False,  # ❌ no augment for val
)

# Training loader with WeightedRandomSampler
_, sw = compute_class_weights(metadata_df)
genre_to_weight = {}
train_full = metadata_df[metadata_df['split'] == 'training']
for genre in GENRE_LABELS:
    count = len(train_full[train_full['genre_top'] == genre])
    genre_to_weight[genre] = len(train_full) / (NUM_CLASSES * max(count, 1))

sample_w = torch.FloatTensor([
    genre_to_weight[train_dataset.df.iloc[i]['genre_top']]
    for i in range(len(train_dataset))
])
sampler = WeightedRandomSampler(sample_w, num_samples=len(train_dataset), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\n✓ Training:   {len(train_dataset)} tracks, {len(train_loader)} batches")
print(f"✓ Validation: {len(val_dataset)} tracks, {len(val_loader)} batches")


In [ ]:
# Fetch 1 batch จาก training + validation
print("Fetching 1 batch from training (with augmentation + scaling)...")
tab_t, spec_t, lab_t = next(iter(train_loader))

print("Fetching 1 batch from validation (no augmentation, with scaling)...")
tab_v, spec_v, lab_v = next(iter(val_loader))

print("\n" + "=" * 65)
print("  FINAL VERIFICATION — FMADatasetV2")
print("=" * 65)

print(f"\n  TRAINING BATCH:")
print(f"    Tabular:      {tab_t.shape}  (expected: ({BATCH_SIZE}, 88))")
print(f"    Spectrogram:  {spec_t.shape}  (expected: ({BATCH_SIZE}, 1, 128, 130))")
print(f"    Labels:       {lab_t.shape}")
print(f"    Tab range:    [{tab_t.min():.2f}, {tab_t.max():.2f}]  (should be ~normalized)")
print(f"    Mel range:    [{spec_t.min():.2f}, {spec_t.max():.2f}]  (should be ~normalized)")

print(f"\n  VALIDATION BATCH:")
print(f"    Tabular:      {tab_v.shape}")
print(f"    Spectrogram:  {spec_v.shape}")
print(f"    Tab range:    [{tab_v.min():.2f}, {tab_v.max():.2f}]")
print(f"    Mel range:    [{spec_v.min():.2f}, {spec_v.max():.2f}]")

# Assertions
assert tab_t.shape == (BATCH_SIZE, 88), f"FAIL: {tab_t.shape}"
assert spec_t.shape == (BATCH_SIZE, 1, 128, 130), f"FAIL: {spec_t.shape}"
assert not torch.isnan(tab_t).any(), "NaN in tabular!"
assert not torch.isnan(spec_t).any(), "NaN in spectrograms!"

print(f"\n  ✅ All assertions PASSED!")
print(f"\n  Pipeline Status:")
print(f"    ✅ Data Cleaning         (corrupted/silent/short removed)")
print(f"    ✅ Feature Engineering    (88 tabular + mel spectrogram)")
print(f"    ✅ Data Augmentation      (noise/stretch/pitch/SpecAugment)")
print(f"    ✅ Feature Scaling        (StandardScaler + global mel norm)")
print(f"    ✅ Class Imbalance        (WeightedRandomSampler + class weights)")
print(f"    ✅ Data Splitting         (FMA official train/val/test)")


### 7.1 Visual Comparison: Training (Augmented) vs Validation (Clean)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i in range(4):
    # Training (augmented)
    axes[0, i].imshow(spec_t[i, 0].numpy(), aspect='auto', origin='lower', cmap='magma')
    genre_t = GENRE_LABELS[lab_t[i].item()]
    axes[0, i].set_title(f'Train: {genre_t}', fontsize=10, fontweight='bold')
    axes[0, i].set_xlabel('Time', fontsize=8)
    axes[0, i].set_ylabel('Freq', fontsize=8)

    # Validation (clean)
    axes[1, i].imshow(spec_v[i, 0].numpy(), aspect='auto', origin='lower', cmap='magma')
    genre_v = GENRE_LABELS[lab_v[i].item()]
    axes[1, i].set_title(f'Val: {genre_v}', fontsize=10, fontweight='bold')
    axes[1, i].set_xlabel('Time', fontsize=8)
    axes[1, i].set_ylabel('Freq', fontsize=8)

axes[0, 0].set_ylabel('Train\n(Augmented)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Val\n(Clean)', fontsize=11, fontweight='bold')

plt.suptitle('Training (Augmented+Scaled) vs Validation (Scaled Only)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'train_vs_val_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ สังเกต: Training spectrograms อาจมี black bands (SpecAugment)")
print("  และ subtle texture changes (noise/stretch/pitch)")
print("  Validation spectrograms จะ clean กว่า")


## 8. 💾 Save Everything for Phase 1B

In [ ]:
# Save class weights
torch.save(class_weights, OUTPUT_DIR / 'class_weights.pt')
print(f"✓ Saved class_weights.pt: {class_weights}")

# Summary of all saved files
print(f"\n{'=' * 65}")
print(f"  ALL SAVED FILES")
print(f"{'=' * 65}")

saved_files = [
    ('cleaned_metadata.csv', 'Cleaned track metadata (from EDA)'),
    ('skip_track_ids.txt', 'Track IDs to skip (from EDA)'),
    ('tabular_scaler.pkl', 'Fitted StandardScaler for 88 tabular features'),
    ('scaling_stats.json', 'Global mel mean/std + tabular stats'),
    ('class_weights.pt', 'Class weights tensor for CrossEntropyLoss'),
    ('augmentation_comparison.png', 'Visual: augmentation effects'),
    ('scaling_verification.png', 'Visual: before/after scaling'),
    ('class_imbalance_handling.png', 'Visual: before/after WeightedRandomSampler'),
    ('train_vs_val_spectrograms.png', 'Visual: augmented vs clean'),
]

for fname, desc in saved_files:
    path = OUTPUT_DIR / fname
    exists = "✓" if path.exists() else "✗"
    size = f"{path.stat().st_size/1024:.1f} KB" if path.exists() else "—"
    print(f"  {exists} {fname:40s} | {size:>10s} | {desc}")


## 9. 📋 Complete Data Preparation Checklist

| # | Step | Status | Details |
|---|------|--------|---------|
| 1 | **Data Collection** | ✅ | FMA Small, 8000 tracks, 8 genres |
| 2 | **Data Cleaning** | ✅ | Corrupted/silent/short track removal |
| 3 | **EDA** | ✅ | Duration, RMS, correlation, F-score, mel stats |
| 4 | **Feature Engineering** | ✅ | 88 tabular features + mel `(1,128,130)` |
| 5 | **Data Transformation** | ✅ | Log-mel + StandardScaler + global normalization |
| 6 | **Data Augmentation** | ✅ | Noise/Stretch/Pitch/SpecAugment (train only) |
| 7 | **Data Splitting** | ✅ | FMA official train/val/test |
| 8 | **Feature Scaling** | ✅ | StandardScaler (tabular) + global norm (mel) |
| 9 | **Class Imbalance** | ✅ | WeightedRandomSampler + class weights |

### ✅ 9/9 COMPLETE — Ready for Phase 1B (Pre-extract .pt files)

### Next: Phase 1B
Pre-extract ทุก features เป็น `.pt` files ครั้งเดียว → training เร็วขึ้น 10-50x
